# Beam torque

A tangential beam pushes the plasma around the torus: the fast ions carry the toroidal angular momentum of the beam, $\dot L = \sum_b w_b\, m R\, v_\varphi$ summed over the births, and hand it to the plasma in two ways, as NUBEAM reports them (Pankin et al., Comput. Phys. Commun. 159, 157, 2004):

- **collisional**: the guiding centre carries $m v_\parallel R b_\varphi$, which the drags give to the electrons and the ions while it slows down. The ions take their energy drag plus all of the pitch-angle scattering, so they get more than their share of the heating;
- **$\mathbf{j}\times\mathbf{B}$**: at ionisation the guiding centre lands a Larmor radius from the birth point. Conservation of the canonical momentum $p_\varphi = m R v_\varphi + q\psi$ fixes the jump of the flux, $q\,\Delta\psi = m R\,(v_\varphi - v_\parallel b_\varphi)$, the gyration's share of the angular momentum. The fast ions' charge crossing the surfaces is a radial current, and the return current of the thermal plasma feels the torque $-\mathbf{j}\cdot\nabla\psi$ between the birth surface and the guiding-centre surface (Honda et al., Nucl. Fusion 49, 035009, 2009).

`beam_torque(deposition, plasma, surfaces, equilibrium)` returns both per flux shell, along $+\varphi$, with the same zero-orbit-width slowing down as `slow_down`.

In [ ]:
import numpy as np
import plasma_plots
import xarray as xr
from plasmacoll.constants import ATOMIC_MASS, ELEMENTARY_CHARGE

import freeofcharge as foc
from freeofcharge.torque import beam_torque

plasma_plots.set_backend("plotly")  # interactive figures, embedded in the docs
plasma_plots.set_math("latex")  # formulas in the figures typeset as LaTeX

equilibrium = foc.Equilibrium.solovev(6.2, 1.8, 1.7, 5.3, wall_psi_n=1.3)
plasma = foc.Plasma.parabolic(1e20, 2e4, ions={"D": (2, 1, 0.96), "He": (4, 2, 0.02)})
rho_edges = np.linspace(0.0, 1.0, 21)
surfaces = equilibrium.flux_surfaces(rho_edges)


def injected(energy, power, *, sense=1, tangency=5.3, markers=500):
    origin, direction = foc.tangential_injection(tangency, 12.0, sense=sense)
    beam = foc.NeutralBeam.gaussian(
        origin,
        direction,
        energy,
        power,
        width=0.3,
        divergence=0.005,
        markers=markers,
        seed=1,
    )
    return foc.deposit(
        beam,
        equilibrium,
        plasma,
        step=0.03,
        births_per_marker=2,
        rho_edges=rho_edges,
        seed=2,
    )


torques = {}
for sense in (1, -1):
    deposition = injected(1.0e6, 16.5e6, sense=sense)
    torques[sense] = beam_torque(deposition, plasma, surfaces, equilibrium)

co = torques[1].attrs
print(f"angular momentum of the births {co['birth_angular_momentum']:.2f} N m")
for name in ("collisional_electrons", "collisional_ions", "jxb"):
    share = co[f"{name}_total"] / co["total"]
    print(f"  {name:22s} {co[f'{name}_total']:6.2f} N m ({100 * share:.0f} %)")
print(f"  total in the shells    {co['total']:6.2f} N m")

## Profiles

The collisional torque lies where the ions are born; the electrons take less than the ions, though the beam heats the electrons more. The $\mathbf{j}\times\mathbf{B}$ torque of a tangential beam is a small part: at its birth an ion moves almost along the field, so its gyration carries little angular momentum.

In [ ]:
names = {
    "collisional_electrons": "collisional, electrons",
    "collisional_ions": "collisional, ions",
    "jxb": "$j \\times B$",
    "total": "total",
}
profiles = (
    torques[1][list(names)].rename(names).to_array("component", name="torque density")
)
profiles.attrs["units"] = "N m/m^3"
profiles.plasma.plot.profiles(x="rho", over="component")

Counter-injection (`sense=-1`) reverses every component: the births lie on the mirror image of the same path, so the profiles agree in magnitude within the sampling noise.

In [ ]:
senses = xr.concat(
    [torques[s].total for s in (1, -1)],
    dim=xr.DataArray(["co", "counter"], dims="injection", name="injection"),
)
senses.name = "torque density"
for name in ("collisional_electrons", "collisional_ions", "jxb"):
    forward, backward = (torques[s].attrs[f"{name}_total"] for s in (1, -1))
    print(f"{name:22s} co {forward:6.2f} N m, counter {backward:6.2f} N m")
    assert backward < 0 < forward and abs(forward / -backward - 1) < 0.05
senses.plasma.plot.profiles(x="rho", over="injection")

## Angular momentum balance

In steady state every unit of angular momentum the births bring ends up in the plasma: the collisional parts add up to the guiding centres' angular momentum (the ions slow down to rest, so none is left to thermalise), and the $\mathbf{j}\times\mathbf{B}$ part is the rest, $\sum_b w_b\, q\,\Delta\psi$. The few births within a Larmor radius of the boundary put part of their $\mathbf{j}\times\mathbf{B}$ torque outside the last closed surface (`jxb_outside`).

In [ ]:
for sense, torque in torques.items():
    a = torque.attrs
    in_plasma = a["total"] + a["jxb_outside"] + a["thermalised_angular_momentum"]
    print(
        f"sense {sense:+d}: births {a['birth_angular_momentum']:7.3f} N m, "
        f"deposited {in_plasma:7.3f} N m, outside {a['jxb_outside']:.3f} N m, "
        f"mismatch {a['balance']:.1e}"
    )
    assert abs(a["balance"]) < 1e-10
    assert abs(a["thermalised_angular_momentum"]) < 1e-4 * abs(a["total"])

How large is the $\mathbf{j}\times\mathbf{B}$ part? For a horizontal beam in the midplane the field has no radial component there and the beam no vertical one, so $v_\parallel \approx v_\varphi b_\varphi$ and the jump is $m R v_\varphi (1 - b_\varphi^2) = m R v_\varphi\, B_p^2 / B^2$: the share is the mean $(B_p / B)^2$ at the births, whatever the tangency radius. It takes a beam tilted out of the midplane, or a stronger poloidal field, to raise it.

In [ ]:
for tangency in (5.3, 4.0, 2.5):
    deposition = injected(1.0e6, 16.5e6, tangency=tangency, markers=300)
    a = beam_torque(deposition, plasma, surfaces, equilibrium).attrs
    x, w = deposition.births.positions, deposition.births.weights
    field = equilibrium.field_at(x)
    b_phi = (x[:, 0] * field[:, 1] - x[:, 1] * field[:, 0]) / np.hypot(x[:, 0], x[:, 1])
    poloidal_share = 1 - b_phi**2 / np.sum(field**2, axis=1)  # (B_p / B)^2
    share = np.sum(w * poloidal_share) / np.sum(w)
    assert abs(a["jxb_total"] / a["total"] - share) < 0.01
    print(
        f"tangency radius {tangency} m: {a['total']:5.2f} N m, "
        f"j x B {100 * a['jxb_total'] / a['total']:.1f} %, "
        f"mean (B_p / B)^2 {100 * share:.1f} %"
    )

## Torque per power and beam energy

A straight line keeps $R\, v_\varphi = v R_t$ ($R_t$ the tangency radius), so a beam brings $\dot L / P = 2 R_t / v$ per watt absorbed: the torque per power falls as $E^{-1/2}$. At a given power a lower energy pushes harder.

In [ ]:
energies = np.array([0.25, 0.5, 1.0, 1.5, 2.0]) * 1e6
per_power = []
for energy in energies:
    deposition = injected(energy, 1.0e6, markers=300)
    a = beam_torque(deposition, plasma, surfaces, equilibrium).attrs
    per_power.append(1e6 * a["total"] / a["birth_power"])
per_power = np.array(per_power)
speed = np.sqrt(2 * energies * ELEMENTARY_CHARGE / (2 * ATOMIC_MASS))
expected = 1e6 * 2 * 5.3 / speed
print("E [MeV]     ", np.round(energies / 1e6, 2))
print("N m per MW  ", np.round(per_power, 3))
print("2 R_t / v   ", np.round(expected, 3))
assert np.allclose(per_power, expected, rtol=0.02)
scan = xr.Dataset(
    {
        "beam_torque": ("energy", per_power),
        "$2 R_t / v$": ("energy", expected),
    },
    coords={"energy": ("energy", energies / 1e3, {"long_name": "$E$", "units": "keV"})},
).to_array("curve", name="torque per absorbed MW")
scan.attrs["units"] = "N m/MW"
scan.plasma.plot.profiles(x="energy", over="curve", logx=True, logy=True)

What the model leaves out: the orbit width (the collisional torque stays on the birth surface, while NUBEAM and ASCOT deposit it along the orbit), the radial current of trapped ions as collisions move their bananas, prompt and later losses and their $\mathbf{j}\times\mathbf{B}$ torque, and charge exchange. The pitch relaxes as in a uniform field, so the angular momentum of ions born trapped goes to the plasma like that of passing ones. On the shells of a 3D equilibrium `beam_torque(..., jxb=False)` gives the collisional part.